# 07 · SEI 상(相) 매핑 (범용) — 있나 / 얼마나 / 어디에

5개 후보상(LiF·Li2O·Li3N·Li2CO3·Li2S)을 **NBD 신호만**으로 스크리닝한다: mean=비정질 halo, max=다결정 링·스팟.
엔진 `fds.analyze_phases(cube)` 한 줄이 **동정(what) + 함량(how-much) + 위치(where) + cepstral 분리**를 다 계산하고,
이 노트북은 그 결과를 **그림으로만** 보여준다(로직은 패키지에). **경로만 바꾸면 어느 데이터셋에도** 그대로.

판정 규칙(정직): **confirmed**=다른 후보가 못 내는 고유 링 보유 / **possible**=일관하나 고유 없음 / **weak/absent**=강한 링 없음.

## 0) 설정 & 로드

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH  = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"  # ★데이터셋 경로만 바꾸면 됨
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)
DET_BIN   = 1                         # 검출기 비닝(메모리). q_max는 안 변함
Q_UNIT_HINT = "1/nm"                 # dm 단위(0.043888 1/nm → /10 = 0.0043888 1/A)
CANDIDATES = ["LiF","Li2O","Li3N","Li2CO3","Li2S"]
VERDICT_COL = {"confirmed":"#2ca02c","possible":"#ff7f0e","weak/absent":"#7f7f7f"}

def _synth(Sy=24,Sx=32,H=96,W=96,seed=0):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/(2*2.0**2))
    halo=lambda r0,s=4:np.exp(-(rr-r0)**2/(2*s**2))
    def spots(r0,n=6,s=1.6,amp=4):
        im=np.zeros((H,W))
        for k in range(n):
            a=2*np.pi*k/n; im+=amp*np.exp(-((xx-cx-r0*np.cos(a))**2+(yy-cy-r0*np.sin(a))**2)/(2*s**2))
        return im
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            base=0.9*beam if iy>=Sy-4 else (beam+spots(22)+0.5*halo(22) if ix<Sx//2 else beam+1.2*halo(20))
            cube[iy,ix]=np.clip(base+0.15*rng.standard_normal((H,W)),0,None)
    return cube

if USE_SYNTHETIC:
    cube=fds.from_array(_synth(), q_per_px=0.02, name="synthetic")
else:
    cube=fds.load(DM4_PATH, Q_UNIT_HINT)
    if DET_BIN>1: cube=fds.bin_cube_detector(cube, DET_BIN)
scan=cube.scan_shape; QPP=cube.calibration.q_per_px
NAME=os.path.splitext(os.path.basename(DM4_PATH))[0] if not USE_SYNTHETIC else "synthetic"
SAVE_DIR=(os.path.dirname(DM4_PATH)+"/nb7_outputs" if not USE_SYNTHETIC else "nb7_outputs"); os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,n): p=os.path.join(SAVE_DIR,f"{NAME}_{n}.png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(n,header,rows):
    import csv; p=os.path.join(SAVE_DIR,f"{NAME}_{n}.csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:",p)
print("cube:",cube.shape,"| q_per_px=",QPP,"| outputs ->",os.path.abspath(SAVE_DIR))

## 0.5) 전처리 진단 — 뭘 교정해야 하나 (측정 먼저)

무작정 교정하면 약한 신호도 버릴 수 있으니, **wander·타원율·검출기 defect를 숫자로 재고** 값이 크면 그때만 교정.
(center 찾기는 hot pixel 제거 후 수행 — 안 그러면 defect가 center를 오염시킴.)

In [ ]:
diag = fds.diagnose_cube(cube)
print('=== 전처리 진단 ===')
print(f"  center = ({diag['center'][0]:.1f},{diag['center'][1]:.1f})")
print(f"  beam wander      = {diag['wander_px']:.2f} px   (>1 이면 per-position 정렬 고려)")
print(f"  detector defects = {100*diag['bad_pixel_frac']:.2f} %  (consistent-hot map으로 안전 수리)")
print(f"  ring ellipticity = {100*diag['ellipticity']:.1f} % @ {diag['ellipse_angle_deg']:.0f}deg  (>2% 면 타원 보정)")
for n in diag['notes']: print('  -',n)
print('\n원칙: 인공물(타원/wander/defect/배경)은 교정 O, 신호 평활(블러/공격적 필터)은 X. hot-pixel은 defect-map만.')

## 1) 엔진 실행 (한 줄) — 동정 + 함량 + 위치 + cepstral

In [ ]:
report = fds.analyze_phases(cube, candidates=CANDIDATES)   # ★ 전부 여기서
report.summary()
material = report.material; center = report.center
diff = report.diffraction
save_csv("verdicts", ["phase","verdict","score","unique_d_A","matched_d_A","missing_strong_d_A","n_spots","amount"],
         [[e.phase,e.verdict,f"{e.score:.3f}","|".join(map(str,e.unique_d)),"|".join(map(str,e.matched_d)),
           "|".join(map(str,e.missing_strong_d)),e.n_spots,f"{e.amount:.4g}"] for e in diff.phases.values()])

## 2) 개요 — median / max / 물질 마스크

In [ ]:
med=fds.median_pattern(cube); mx=np.asarray(cube.max_dp(),float)
fig,ax=plt.subplots(1,3,figsize=(13,4.2))
ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(*center,"c+",ms=9); ax[0].set_title("median (amorphous halo)"); ax[0].axis("off")
ax[1].imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="magma"); ax[1].plot(*center,"c+",ms=9); ax[1].set_title("MAX (polycrystal rings/spots)"); ax[1].axis("off")
ax[2].imshow(material,cmap="gray"); ax[2].set_title(f"material mask ({100*material.mean():.0f}%)"); ax[2].axis("off")
plt.tight_layout(); save(fig,"01_overview"); plt.show()

## 3) 동정(what) — 링 인덱싱 + 상별 판정

In [ ]:
qm,Im=fds.azimuthal_integrate(fds.clean_pattern(mx,hot_threshold=8.0),center,q_per_px=QPP)
fig,ax=plt.subplots(1,2,figsize=(14,4.6))
ax[0].semilogy(qm,np.clip(Im,1e-2,None),"k-",lw=0.8)
for d in diff.rings_d:
    if d>0: ax[0].axvline(1/d,color="r",ls=":",lw=0.8)
for c in CANDIDATES:
    for dd,w in fds.COMPOUND_RINGS[c]: ax[0].axvline(1/dd,color="0.7",ls="--",lw=0.3+0.7*w,alpha=0.4)
ax[0].set_xlabel("q (1/A)"); ax[0].set_ylabel("MAX I(q)"); ax[0].set_title(f"rings d(A)={[round(d,2) for d in diff.rings_d]}")
ph=list(diff.phases.values())
ax[1].barh([e.phase for e in ph],[e.score for e in ph],color=[VERDICT_COL[e.verdict] for e in ph])
for i,e in enumerate(ph): ax[1].text(e.score+0.01,i,e.verdict+(" *"+str(e.unique_d) if e.unique_d else ""),va="center",fontsize=8)
ax[1].set_xlim(0,1.25); ax[1].set_xlabel("ring-match score"); ax[1].set_title("phase verdict (green=confirmed, orange=possible, gray=weak)")
plt.tight_layout(); save(fig,"03_identity"); plt.show()
print("amorphous FSDP: q=%.3f d=%.2f A conf %.1f | crystallinity %.1f | unexplained d=%s"%(
    diff.halo_q,1/diff.halo_q if diff.halo_q>0 else 0,diff.halo_conf,diff.crystallinity,diff.unexplained_d))

## 4) 어디에(where) — 상별 위치 지도 (두께정규화 DF) + 함량

In [ ]:
maps=report.location_maps; ph=list(diff.phases.values())
fig,ax=plt.subplots(1,len(ph),figsize=(3.2*len(ph),3.6))
for a,e in zip(np.atleast_1d(ax),ph):
    m=maps.get(e.phase)
    if m is None: a.set_title(f"{e.phase}\n(no ring)",fontsize=8); a.axis("off"); continue
    im=a.imshow(np.where(material,m,np.nan),cmap="inferno"); a.axis("off")
    a.set_title(f"{e.phase} [{e.verdict}]\nd={e.diag_d:.2f}A amt={e.amount:.3g}",fontsize=8); plt.colorbar(im,ax=a,fraction=0.046)
fig.suptitle("per-phase location (thickness-normalized DF at diagnostic ring)")
plt.tight_layout(); save(fig,"04_location_maps"); plt.show()

## 5) 몇 개 구조 / 어디에 (cepstral) — **상별 특징 거리에서** fluctuation 위치

per-phase NBD 위치지도처럼, **각 상의 특징 원자간 거리**(LiF 2.01·Li2O 2.00·Li3N 1.94·Li2CO3 1.28·Li2S 2.47 Å)에서
cepstral fluctuation을 잡아 **상별 공간 위치**를 본다. **주의**: LiF/Li2O/Li3N은 nn~2.0Å 겹쳐 지도가 유사(분해능 한계),
**Li2CO3(1.28)·Li2S(2.47)만 거리로 뚜렷 구분**. 아래 참고로 일반 거리밴드도 같이.

In [ ]:
cepP=report.cepstral_phase_maps; cep=report.cepstral_bands
from fourdstem import PHASE_DISTANCE
ph=list(diff.phases.values())
fig,ax=plt.subplots(1,len(ph),figsize=(3.2*len(ph),3.6))
for a,e in zip(np.atleast_1d(ax),ph):
    m=cepP.get(e.phase) if cepP else None
    if m is None: a.set_title(f"{e.phase}\n(n/a)",fontsize=8); a.axis("off"); continue
    mm=m.reshape(scan) if m.ndim==1 else m
    im=a.imshow(np.where(material,mm,np.nan),cmap="viridis"); a.axis("off")
    a.set_title(f"{e.phase} [{e.verdict}]\ncepstral r~{PHASE_DISTANCE[e.phase]:.2f}A",fontsize=8); plt.colorbar(im,ax=a,fraction=0.046)
fig.suptitle("per-phase cepstral location (fluctuation at each phase's diagnostic interatomic distance)")
plt.tight_layout(); save(fig,"05_cepstral_phase"); plt.show()
print("주의: LiF/Li2O/Li3N은 nn~2.0A 겹쳐 지도 유사(분해능 한계). Li2CO3(1.28)/Li2S(2.47)만 거리로 뚜렷 구분.")
if cep is not None:
    fb=report.fbands
    fig,ax=plt.subplots(1,len(fb),figsize=(4.0*len(fb),3.4))
    for a,(band,m) in zip(np.atleast_1d(ax),zip(fb,cep)):
        mm=np.asarray(m,float); mm=mm.reshape(scan) if mm.ndim==1 else mm
        im=a.imshow(np.where(material,mm,np.nan),cmap="viridis"); a.axis("off"); a.set_title(f"band {band[0]}-{band[1]} A",fontsize=9); plt.colorbar(im,ax=a,fraction=0.046)
    fig.suptitle("generic distance bands (reference)"); plt.tight_layout(); save(fig,"05b_cepstral_bands"); plt.show()

## 6) 정직한 요약

In [ ]:
conf=[e.phase for e in diff.phases.values() if e.verdict=="confirmed"]
poss=[e.phase for e in diff.phases.values() if e.verdict=="possible"]
weak=[e.phase for e in diff.phases.values() if e.verdict=="weak/absent"]
print(f"=== {NAME} 요약 ===")
print(f"  확정(confirmed, 고유 링): {conf or '없음'}")
print(f"  가능(possible, 일관하나 겹침): {poss or '없음'}")
print(f"  약함/없음(weak): {weak or '없음'}")
print(f"  미설명 링 d(A): {diff.unexplained_d}  (후보 5상 밖 = 이름 못 붙인 상 가능)")
print("\n[한계]")
print("  - 회절 링/스팟은 확정에 강하나, ~2A 상(LiF/Li2O/Li3N)은 수렴각 분해능으로 겹쳐 특정 제한.")
print("  - cepstral은 구조가 다른 영역을 '분리·매핑'하나 상 이름은 안 붙임 → 링 인덱싱 + EDS로 보강.")
print("  - RDF는 이 카메라길이(Q_max~7)엔 저해상도 → 사용 안 함. 링/스팟/cepstral로 진행.")